# Create Velux Stiftung Awards (GRANT PATTERN, hybrid REST + HTML)

Velux Stiftung projects from the foundation's own WordPress at veluxstiftung.ch. Switzerland-based Velux foundation (distinct from the Danish Velux Fonden), funding Daylight Research, Forestry, Healthy Ageing, and Ophthalmology.

**Prerequisites:**
- Run `scripts/local/velux_stiftung_to_s3.py` first.

**Data source (hybrid):**
- `/wp/v2/projects` REST endpoint (31 projects total) — title, slug, content, projects_type
- Per-project detail page (~110KB) — structured "Funding amount: CHF N" + "YYYY - YYYY" period labels

**S3 location:** `s3a://openalex-ingest/awards/velux_stiftung/velux_stiftung_projects.parquet`

**Awarding body:**
- funder_id: 4320309607
- display_name: "Velux Stiftung"
- country: CH (Switzerland)
- ROR: (not exposed in OpenAlex record)
- DOI: 10.13039/100007214

**Coverage (full local run, 2026-05-22):**
- 31 projects total
- 100% title/slug/description/start_year/end_year/type_names
- 93.5% amount (29/31 projects have CHF amount on detail page)
- Total CHF 14.4M across rows with amount; min CHF 106k, max CHF 2.95M
- 31/31 unique slug → funder_award_id (`velux-stiftung-{slug}`)

**Funding areas (from `projects_type` taxonomy):**
- Daylight Research
- Healthy Ageing
- Forestry
- Ophthalmology
- Lighthouse projects (cross-cutting)

**Currency:** CHF hardcoded (Swiss foundation, only currency the source publishes).

**Provenance:** `velux_stiftung_projects` (direct from foundation, not an aggregator).

**Priority:** 115 (UCOP 106 / Rita Allen 107 / Schmidt Sciences 108 / NOMIS 109 / Wenner-Gren 110 / KAW 111 / Helmsley 112 / Mott 113 / Gulbenkian 114 are immediately-prior slots in flight).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.velux_stiftung_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/velux_stiftung/velux_stiftung_projects.parquet`;


In [ ]:
%sql SELECT COUNT(*) as total_rows FROM openalex.awards.velux_stiftung_raw; 

In [ ]:
%sql DESCRIBE openalex.awards.velux_stiftung_raw; 

## Step 1.5: Inspect Raw Data, Money Scan, Native Key

In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.velux_stiftung_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.velux_stiftung_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT project_id, slug, title, type_names, amount, currency, start_year, end_year
FROM openalex.awards.velux_stiftung_raw LIMIT 10;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)) / 1e6, 2) AS total_chf_millions
FROM openalex.awards.velux_stiftung_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(DISTINCT project_id) AS distinct_project_ids
FROM openalex.awards.velux_stiftung_raw;


In [ ]:
%sql
SELECT start_year, COUNT(*) as projects, ROUND(SUM(TRY_CAST(amount AS DOUBLE))/1e6,2) as total_chf_m
FROM openalex.awards.velux_stiftung_raw
GROUP BY start_year ORDER BY start_year;


In [ ]:
%sql
SELECT type_names, COUNT(*) as cnt
FROM openalex.awards.velux_stiftung_raw
GROUP BY type_names ORDER BY cnt DESC;


## Step 1.6: Fail-fast — Verify the Velux Stiftung Funder Row Exists

**Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320309607;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.velux_stiftung_awards
USING delta
AS
WITH
vlx_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320309607
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(start_year AS INT) AS parsed_start_year,
        TRY_CAST(end_year AS INT) AS parsed_end_year,
        TRY_TO_DATE(CONCAT(start_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(CONCAT(end_year, '-12-31'), 'yyyy-MM-dd') AS parsed_end_date
    FROM openalex.awards.velux_stiftung_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        g.parsed_amount as amount,
        g.currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,
        COALESCE(NULLIF(TRIM(g.type_names), ''), 'Velux Stiftung Project') as funder_scheme,

        'velux_stiftung_projects' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_start_year as start_year,
        g.parsed_end_year as end_year,

        -- lead_investigator: Velux Stiftung doesn't publish PI structurally;
        -- per HHMI/Helmsley/Mott precedent for org-level grants, leave NULL.
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.link as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN vlx_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 115

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'velux_stiftung_projects' AND priority = 115;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    115 as priority
FROM openalex.awards.velux_stiftung_awards;


## Verification Queries

In [ ]:
%sql SELECT COUNT(*) as total FROM openalex.awards.velux_stiftung_awards;

In [ ]:
%sql DESCRIBE openalex.awards.velux_stiftung_awards;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date
FROM openalex.awards.velux_stiftung_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, end_year, amount, currency, funder_scheme, landing_page_url
FROM openalex.awards.velux_stiftung_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt FROM openalex.awards.velux_stiftung_awards GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as projects, ROUND(SUM(amount)/1e6,2) as total_chf_m
FROM openalex.awards.velux_stiftung_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year;


In [ ]:
%sql
-- §6.7 Expected: ~93.5% pct_amount, CHF only
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(SUM(amount) / 1e6, 2) AS total_chf_millions
FROM openalex.awards.velux_stiftung_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt FROM openalex.awards.velux_stiftung_awards GROUP BY funder_scheme ORDER BY cnt DESC;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.velux_stiftung_awards;
